# ASR Shootout — Clean Pipeline

Benchmark of 5 ASR systems on:
1. **Your 20 self-recorded** Bangalore-locality clips (primary; metric = entity accuracy)
2. **Common Voice Hindi** (open-source; 20 samples; metric = WER/CER)
3. **Kathbath Hindi** (open-source; 20 samples; metric = WER/CER)

**Models (5 distinct approaches):**
- Deepgram Nova-3 (commercial, global baseline; multilingual code-switching)
- Sarvam Saarika v2.5 (commercial, India-specific — direct Deepgram competitor)
- Whisper large-v3 (open, multilingual, large)
- IndicConformer-600M-Multi (open, India-specific, trained on all 22 Indian langs incl. Kannada)
- IndicWav2Vec / Vakyansh Hindi (open, India-specific, Hindi-only — older wav2vec2 baseline)

**Language handling:** each clip's true language is routed to every model that accepts a
language argument, so Kannada is decoded as Kannada (not Hindi-forced). This removes the
earlier Kannada handicap.

Run cells **top to bottom** with Shift+Enter. Set **Runtime → Change runtime type → T4 GPU** first.
Fixes baked in: Devanagari→Roman transliteration, wav2vec2 token cleanup, Deepgram v3/v4, mp4 support.


## 1 · Install dependencies (~4 min)

In [ ]:
# Pin datasets to 2.x so Common Voice's loader script still works
# (datasets 3.x removed support for dataset loading scripts).
!pip install -q "deepgram-sdk==3.7.0" faster-whisper transformers torch torchaudio \
                librosa jiwer rapidfuzz pandas matplotlib soundfile "datasets==2.20.0" \
                indic-transliteration requests
print("Install complete.")

## 2 · Hugging Face login (one time)

Kathbath and Common Voice are gated. You must:
1. Have a free account at huggingface.co
2. Accept terms on each dataset page (open these once in your browser):
   - https://huggingface.co/datasets/mozilla-foundation/common_voice_17_0
   - https://huggingface.co/datasets/ai4bharat/Kathbath
3. Create a token at https://huggingface.co/settings/tokens (Read access)
4. Paste it below.

In [ ]:
from huggingface_hub import login
login()   # paste your hf_... token in the box that appears

## 3 · API keys (Deepgram + Sarvam) + GPU check

- Deepgram key: from deepgram.com console (free $200 credit)
- Sarvam key: sign up at dashboard.sarvam.ai, copy your API subscription key (free credits)

Paste both below.

In [ ]:
import os, torch
os.environ['DEEPGRAM_API_KEY'] = 'PASTE_YOUR_DEEPGRAM_KEY_HERE'
os.environ['SARVAM_API_KEY']   = 'PASTE_YOUR_SARVAM_KEY_HERE'
print("Deepgram key set:", os.environ['DEEPGRAM_API_KEY'] != 'PASTE_YOUR_DEEPGRAM_KEY_HERE')
print("Sarvam key set:  ", os.environ['SARVAM_API_KEY'] != 'PASTE_YOUR_SARVAM_KEY_HERE')
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU - set Runtime to T4 GPU!")

## 4 · Shared helpers — metrics, transliteration, normalization

Defined once and reused everywhere (single source of truth for scoring).
Transliteration covers **both Hindi (Devanagari) and Kannada** scripts → Roman,
so a model that correctly transcribes Kannada in Kannada script is matched fairly
against the Roman-script locality names (not penalized for the script).

In [ ]:
import re, time
import pandas as pd
from functools import lru_cache
import jiwer
from rapidfuzz import fuzz
from indic_transliteration import sanscript
from indic_transliteration.sanscript import transliterate

# Script ranges. We transliterate BOTH Hindi (Devanagari) and Kannada to Roman
# before matching, so a correct native-script transcription is not unfairly
# penalized against a Roman-script locality database.
DEV = re.compile(r'[\u0900-\u097F]')   # Devanagari (Hindi)
KAN = re.compile(r'[\u0C80-\u0CFF]')   # Kannada

def clean(t):
    """Strip wav2vec2 special tokens like <s>, </s>."""
    if not isinstance(t, str): return ""
    t = re.sub(r'</?s>', '', t)
    t = re.sub(r'<[^>]*>', '', t)
    return re.sub(r'\s+', ' ', t).strip()

def to_roman(t):
    """Transliterate Devanagari OR Kannada script to Roman (ITRANS) for fair matching."""
    if not isinstance(t, str) or not t.strip(): return ""
    try:
        if DEV.search(t):
            t = transliterate(t, sanscript.DEVANAGARI, sanscript.ITRANS)
        elif KAN.search(t):
            t = transliterate(t, sanscript.KANNADA, sanscript.ITRANS)
    except Exception:
        pass
    return t

def normalize(t):
    t = to_roman(clean(t)).lower()
    t = re.sub(r'[^a-z0-9 ]', ' ', t)
    return re.sub(r'\s+', ' ', t).strip()

def compute_wer(ref, hyp):
    r, h = normalize(ref), normalize(hyp)
    if not r: return 1.0 if h else 0.0
    try: return jiwer.wer(r, h)
    except Exception: return 1.0

def compute_cer(ref, hyp):
    r, h = normalize(ref), normalize(hyp)
    if not r: return 1.0 if h else 0.0
    try: return jiwer.cer(r, h)
    except Exception: return 1.0

def entity_match(locality, hyp, thr=80):
    """Did the locality survive transcription? Fuzzy, cross-script aware (Hindi + Kannada)."""
    nl, nh = normalize(locality), normalize(hyp)
    if not nh: return False, 0
    s = fuzz.partial_ratio(nl, nh)
    return s >= thr, int(s)


import librosa as _lb
def audio_duration_seconds(path):
    """Duration of an audio file in seconds (for Real-Time Factor)."""
    try:
        return _lb.get_duration(path=path)
    except Exception:
        try:
            y, sr = _lb.load(path, sr=None, mono=True); return len(y)/sr
        except Exception:
            return None

def entity_char_distance(locality, hyp):
    """Near-miss quality: 0=perfect, 1=totally wrong. Low=recoverable by fuzzy lookup."""
    nl, nh = normalize(locality), normalize(hyp)
    if not nh: return 1.0
    score = fuzz.partial_ratio(nl, nh)
    return round(1.0 - score/100.0, 3)

print("Helpers ready: Kannada transliteration + RTF + entity-distance.")

## 5 · The 4 transcribers

Each returns a dict: {transcript, latency_seconds}. Models are loaded once and cached.

In [ ]:
# Language code maps — route each clip's true language to each model.
# Models that accept a language hint get the right one; this fixes the Kannada handicap.
WHISPER_LANG = {"hindi":"hi","hinglish":"hi","kannada":"kn","english":"en"}
INDIC_LANG   = {"hindi":"hi","hinglish":"hi","kannada":"kn","english":"hi"}  # IndicConformer/wav2vec: Indic codes
SARVAM_LANG  = {"hindi":"hi-IN","hinglish":"hi-IN","kannada":"kn-IN","english":"en-IN"}
DEEPGRAM_LANG= {"hindi":"multi","hinglish":"multi","kannada":"multi","english":"multi"}  # Nova-3 multilingual code-switch; no native Kannada

# ---- Deepgram (API, v3/v4 compatible) ----
def deepgram_transcribe(audio_path, language="multi", model="nova-3"):
    from deepgram import DeepgramClient
    client = DeepgramClient(os.environ["DEEPGRAM_API_KEY"])
    with open(audio_path, "rb") as f: buf = f.read()
    start = time.perf_counter(); resp = None
    try:
        resp = client.listen.v1.media.transcribe_file(
            request=buf, model=model, language=language, smart_format=True, punctuate=True)
    except (AttributeError, TypeError):
        resp = None
    if resp is None:
        from deepgram import PrerecordedOptions
        opts = PrerecordedOptions(model=model, language=language, smart_format=True, punctuate=True)
        resp = client.listen.rest.v("1").transcribe_file({"buffer": buf}, opts)
    lat = time.perf_counter() - start
    try: tx = resp.results.channels[0].alternatives[0].transcript
    except Exception:
        d = resp.to_dict() if hasattr(resp,"to_dict") else resp
        try: tx = d["results"]["channels"][0]["alternatives"][0]["transcript"]
        except Exception: tx = ""
    return {"transcript": tx, "latency_seconds": round(lat,3)}

# ---- Sarvam Saarika (API, India-specific, Deepgram competitor) ----
import requests
def sarvam_transcribe(audio_path, language_code="unknown", model="saarika:v2.5"):
    url = "https://api.sarvam.ai/speech-to-text"
    headers = {"api-subscription-key": os.environ["SARVAM_API_KEY"]}
    start = time.perf_counter()
    with open(audio_path, "rb") as f:
        files = {"file": (os.path.basename(audio_path), f, "audio/wav")}
        data = {"model": model, "language_code": language_code}
        r = requests.post(url, headers=headers, files=files, data=data, timeout=60)
    lat = time.perf_counter() - start
    try: tx = r.json().get("transcript", "")
    except Exception: tx = ""
    return {"transcript": tx, "latency_seconds": round(lat,3)}

# ---- Whisper large-v3 (faster-whisper, multilingual) ----
from faster_whisper import WhisperModel
@lru_cache(maxsize=2)
def _wh(size):
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    ct  = "int8_float16" if dev=="cuda" else "int8"
    try: return WhisperModel(size, device=dev, compute_type=ct)
    except Exception: return WhisperModel(size, device="cpu", compute_type="int8")

def whisper_transcribe(audio_path, size="large-v3", language="hi"):
    model = _wh(size)
    start = time.perf_counter()
    segs, _ = model.transcribe(audio_path, language=language, beam_size=5)
    tx = " ".join(s.text.strip() for s in segs)
    return {"transcript": tx.strip(), "latency_seconds": round(time.perf_counter()-start,3)}

# ---- IndicConformer-600M-Multi (open, all 22 Indian langs, per-language decoding) ----
import torchaudio
@lru_cache(maxsize=1)
def _conformer():
    from transformers import AutoModel
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    m = AutoModel.from_pretrained("ai4bharat/indic-conformer-600m-multilingual",
                                  trust_remote_code=True).to(dev).eval()
    return m, dev

def indicconformer_transcribe(audio_path, language="hi", decoding="ctc"):
    model, dev = _conformer()
    wav, sr = torchaudio.load(audio_path)
    wav = torch.mean(wav, dim=0, keepdim=True)
    if sr != 16000:
        wav = torchaudio.transforms.Resample(sr, 16000)(wav)
    start = time.perf_counter()
    with torch.no_grad():
        tx = model(wav.to(dev), language, decoding)
    if isinstance(tx, (list, tuple)): tx = tx[0] if tx else ""
    return {"transcript": str(tx).strip(), "latency_seconds": round(time.perf_counter()-start,3)}

# ---- IndicWav2Vec (Vakyansh Hindi, open, Hindi-only) ----
import librosa
@lru_cache(maxsize=1)
def _w2v():
    from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor
    mid = "Harveenchadha/vakyansh-wav2vec2-hindi-him-4200"
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    proc = Wav2Vec2Processor.from_pretrained(mid)
    mdl  = Wav2Vec2ForCTC.from_pretrained(mid).to(dev).eval()
    return mdl, proc, dev

def indicwav2vec_transcribe(audio_path, language="hi"):
    mdl, proc, dev = _w2v()
    audio, _ = librosa.load(audio_path, sr=16000, mono=True)
    iv = proc(audio, sampling_rate=16000, return_tensors="pt").input_values.to(dev)
    start = time.perf_counter()
    with torch.no_grad(): logits = mdl(iv).logits
    ids = torch.argmax(logits, dim=-1)
    tx = proc.batch_decode(ids)[0]
    return {"transcript": clean(tx), "latency_seconds": round(time.perf_counter()-start,3)}

# MODELS take (audio_path, lang_label) so we can route the correct language per clip.
# lang_label is one of: hindi / hinglish / kannada / english
MODELS = {
    "deepgram_nova3":   lambda p, lang: deepgram_transcribe(p, language=DEEPGRAM_LANG.get(lang,"multi")),
    "sarvam_saarika":   lambda p, lang: sarvam_transcribe(p, language_code=SARVAM_LANG.get(lang,"unknown")),
    "whisper_large_v3": lambda p, lang: whisper_transcribe(p, size="large-v3", language=WHISPER_LANG.get(lang,"hi")),
    "indicconformer":   lambda p, lang: indicconformer_transcribe(p, language=INDIC_LANG.get(lang,"hi")),
    "indicwav2vec":     lambda p, lang: indicwav2vec_transcribe(p, language="hi"),  # Hindi-only by design
}
print("Transcribers ready:", list(MODELS.keys()))

## 6 · Your own recordings — upload + convert

Upload your 20 clips (any format incl. .mp4). They are converted to 16kHz mono wav.
Filenames must follow: `NN_locality_condition_language.ext`

In [ ]:
from google.colab import files
import os, soundfile as sf

os.makedirs('recordings_raw', exist_ok=True)
os.makedirs('recordings', exist_ok=True)
up = files.upload()  # select all 20 audio files
for fn in list(up.keys()):
    os.rename(fn, os.path.join('recordings_raw', fn))

AUDIO_EXTS = {'.m4a','.mp3','.wav','.ogg','.flac','.aac','.opus','.mp4','.mov','.3gp','.amr','.webm'}
for f in sorted(os.listdir('recordings_raw')):
    ext = os.path.splitext(f)[1].lower()
    if ext not in AUDIO_EXTS: continue
    audio, _ = librosa.load(os.path.join('recordings_raw', f), sr=16000, mono=True)
    out = os.path.splitext(f)[0] + '.wav'
    sf.write(os.path.join('recordings', out), audio, 16000)
    print(f"converted {f} -> {out}")
print("\nDone. Files in recordings/:", len(os.listdir('recordings')))

## 7 · Ground truth for your recordings

Loads `ground_truth.csv` from your project zip (filename, locality, reference, condition, language).
If you didn't unzip the project, upload `ground_truth.csv` first with the small uploader cell below.

In [ ]:
# OPTIONAL: only run this if ground_truth.csv is NOT already on disk.
# It lets you upload the CSV directly.
import os
if not os.path.exists('ground_truth.csv') and not __import__('glob').glob('/content/**/ground_truth.csv', recursive=True):
    from google.colab import files
    up = files.upload()   # choose ground_truth.csv
    print("Uploaded:", list(up.keys()))
else:
    print("ground_truth.csv already present — skip this cell.")

In [ ]:
# Load ground truth from CSV (instead of hardcoding it here).
# This file ships in your project zip. The cell finds it wherever it sits.
import os, glob, pandas as pd

gt_path = None
for cand in ['ground_truth.csv'] + glob.glob('/content/**/ground_truth.csv', recursive=True):
    if os.path.exists(cand):
        gt_path = cand
        break

assert gt_path is not None, "ground_truth.csv not found. Upload it or unzip your project first."
print("Loading ground truth from:", gt_path)

gt = pd.read_csv(gt_path)
# keep only rows whose audio file is actually present in recordings/
present = set(os.listdir('recordings')) if os.path.exists('recordings') else set()
gt = gt[gt['filename'].isin(present)].reset_index(drop=True)
print(f"{len(gt)} of {sum(1 for _ in open(gt_path))-1} recordings present and matched.")
gt.head()

## 8 · Run all models on YOUR recordings → entity accuracy

This is the primary evaluation. Metric: did the locality survive transcription?

In [ ]:
import traceback
rows = []
for mname, fn in MODELS.items():
    print(f"\n=== {mname} ===")
    for _, r in gt.iterrows():
        path = os.path.join('recordings', r['filename'])
        dur = audio_duration_seconds(path)
        try:
            res = fn(path, r['language'])
            m, s = entity_match(r['locality'], res['transcript'])
            lat = res['latency_seconds']
            rtf = round(lat/dur, 3) if (dur and dur > 0) else None
            rows.append({"dataset":"own","model":mname,"item":r['locality'],
                         "condition":r['condition'],"language":r['language'],
                         "reference":r['reference'],"transcript":res['transcript'],
                         "audio_seconds":round(dur,3) if dur else None,
                         "latency_seconds":lat,"rtf":rtf,
                         "wer":round(compute_wer(r['reference'],res['transcript']),3),
                         "cer":round(compute_cer(r['reference'],res['transcript']),3),
                         "entity_matched":m,"entity_score":s,
                         "entity_distance":entity_char_distance(r['locality'],res['transcript'])})
            print(f"  {r['locality']:<20} ({r['language']:<8}) match={m} rtf={rtf}")
        except Exception as e:
            print(f"  ERROR {r['locality']}: {str(e)[:90]}")
own_df = pd.DataFrame(rows)
own_df.to_csv('results_own.csv', index=False)
print("\nSaved results_own.csv —", len(own_df), "rows")


## 9 · Load open-source datasets (Common Voice + Kathbath)

We take 50 samples from each. Metric here is WER/CER (these are general Hindi
sentences, not locality names, so entity accuracy doesn't apply).

If a dataset fails to load (gating/version), the cell prints the error and skips it —
that itself is a valid finding to note.

In [ ]:
from datasets import load_dataset
N_SAMPLES = 20
open_clips = []   # list of (dataset_name, audio_array, sampling_rate, reference_text)

# ---------- Common Voice Hindi ----------
# CV 17 uses a loader script that new `datasets` rejects. We try a few options and
# use whichever works. (If all fail, we just skip CV and report it honestly.)
cv_loaded = 0
def _try_cv(repo, **kw):
    global cv_loaded
    try:
        ds = load_dataset(repo, "hi", split="test", streaming=True, **kw)
        cnt = 0
        for ex in ds:
            audio = ex.get("audio") or ex.get("path")
            arr = audio["array"] if isinstance(audio, dict) else None
            sr  = audio["sampling_rate"] if isinstance(audio, dict) else None
            sent = ex.get("sentence") or ex.get("text") or ""
            if arr is None: continue
            open_clips.append(("common_voice", arr, sr, sent))
            cnt += 1
            if cnt >= N_SAMPLES: break
        cv_loaded = cnt
        return cnt > 0
    except Exception as e:
        print(f"  CV via {repo}: {str(e)[:120]}")
        return False

if not _try_cv("mozilla-foundation/common_voice_17_0", trust_remote_code=True):
    if not _try_cv("mozilla-foundation/common_voice_13_0", trust_remote_code=True):
        _try_cv("fsicoli/common_voice_17_0", trust_remote_code=True)
print(f"Common Voice: loaded {cv_loaded} samples")

# ---------- Kathbath Hindi ----------
# Correct fields: transcript is in `text`, audio in `audio_filepath`.
kb_loaded = 0
try:
    kb = load_dataset("ai4bharat/Kathbath", "hindi", split="valid", streaming=True)
    cnt = 0
    for ex in kb:
        audio = ex.get("audio_filepath") or ex.get("audio")
        text  = ex.get("text") or ex.get("transcript") or ""
        if audio is None or not text: continue
        # audio_filepath is an AudioDecoder/dict exposing array + sampling_rate
        if isinstance(audio, dict):
            arr, sr = audio["array"], audio["sampling_rate"]
        else:
            # torchcodec AudioDecoder: get_all_samples()
            try:
                samples = audio.get_all_samples()
                arr = samples.data.squeeze().cpu().numpy()
                sr  = samples.sample_rate
            except Exception:
                continue
        open_clips.append(("kathbath", arr, sr, text))
        cnt += 1
        if cnt >= N_SAMPLES: break
    kb_loaded = cnt
except Exception as e:
    print("  Kathbath error:", str(e)[:160])
print(f"Kathbath: loaded {kb_loaded} samples")

print("\nTotal open-source clips:", len(open_clips))

## 10 · Run models on open-source data → WER/CER

We write each clip to a temp wav then run every model. (Deepgram is an API call per clip;
if you want to save credits, comment Deepgram out of LOCAL_MODELS below.)

In [ ]:
import soundfile as sf, tempfile, numpy as np, librosa as lb

OPEN_MODELS = dict(MODELS)   # all 5; comment out Deepgram/Sarvam here to save API credits

open_rows = []
for i, (dsname, arr, sr, ref) in enumerate(open_clips):
    tmp = tempfile.mktemp(suffix=".wav")
    a = np.asarray(arr, dtype="float32")
    if sr != 16000:
        a = lb.resample(a, orig_sr=sr, target_sr=16000)
    sf.write(tmp, a, 16000)
    for mname, fn in OPEN_MODELS.items():
        try:
            res = fn(tmp, "hindi")   # open datasets here are Hindi
            open_rows.append({"dataset":dsname,"model":mname,"item":f"{dsname}_{i}",
                              "reference":ref,"transcript":res["transcript"],
                              "latency_seconds":res["latency_seconds"],
                              "wer":round(compute_wer(ref,res["transcript"]),3),
                              "cer":round(compute_cer(ref,res["transcript"]),3)})
        except Exception as e:
            print(f"  ERROR {dsname}#{i} {mname}: {str(e)[:80]}")
    if (i+1) % 10 == 0: print(f"  processed {i+1}/{len(open_clips)} clips")
    os.remove(tmp)

open_df = pd.DataFrame(open_rows)
open_df.to_csv('results_open.csv', index=False)
print("\nSaved results_open.csv —", len(open_df), "rows")

## 11 · Results

Two tables:
- **Own recordings:** entity accuracy + WER/CER + latency, sliced by condition and language
- **Open data:** WER/CER per model per dataset

In [ ]:
print("="*60)
print("A) YOUR RECORDINGS — primary metric: entity accuracy")
print("="*60)
agg_own = own_df.groupby("model").agg(
    entity_accuracy=("entity_matched","mean"),
    wer_mean=("wer","mean"), cer_mean=("cer","mean"),
    latency_p50=("latency_seconds","median"),
    n=("item","count")).round(3).sort_values("entity_accuracy", ascending=False)
display(agg_own)
agg_own.to_csv("agg_own.csv")

print("\nEntity accuracy by CONDITION:")
display(own_df.pivot_table(index="model", columns="condition", values="entity_matched", aggfunc="mean").round(2))
print("\nEntity accuracy by LANGUAGE:")
display(own_df.pivot_table(index="model", columns="language", values="entity_matched", aggfunc="mean").round(2))

if len(open_df):
    print("\n" + "="*60)
    print("B) OPEN-SOURCE DATA — metric: WER / CER (lower = better)")
    print("="*60)
    agg_open = open_df.groupby(["dataset","model"]).agg(
        wer_mean=("wer","mean"), cer_mean=("cer","mean"),
        latency_p50=("latency_seconds","median"), n=("item","count")).round(3)
    display(agg_open)
    agg_open.to_csv("agg_open.csv")
else:
    print("\n(No open-source data loaded — skipping section B)")

## 11b · Production decision metrics (RTF · near-miss quality)

Two metrics that drive model choice for a live telephony product, beyond raw accuracy:
- **Real-Time Factor (RTF)** = processing time / audio length. <1 = faster than real-time (can keep up with live calls); lower scales cheaper.
- **Mean entity distance on misses** = when the locality is missed, how close was it? 0 = perfect, 1 = totally wrong. Low values mean graceful failures recoverable by fuzzy-matching against the known-localities list; high values mean catastrophic, confidently-wrong outputs.


In [ ]:
import numpy as np

prod = own_df.groupby("model").agg(
    rtf_p50=("rtf","median"),
    rtf_p95=("rtf", lambda s: s.quantile(0.95)),
).round(3)

# Mean entity distance ON MISSES only: how badly does a model fail when it fails?
# 0 = perfect, 1 = totally wrong. Low = recoverable by fuzzy lookup (graceful);
# high = catastrophic (confidently wrong place name).
miss = own_df[~own_df["entity_matched"]]
prod["miss_entity_distance"] = miss.groupby("model")["entity_distance"].mean().round(3)
prod["n_misses"] = miss.groupby("model")["item"].count()

prod = prod.sort_values("rtf_p50")
print("=== Production decision metrics (RTF + near-miss quality) ===")
display(prod)
prod.to_csv("prod_metrics.csv")


## 12 · Charts

In [ ]:
import matplotlib.pyplot as plt

# Entity accuracy bar (own recordings)
fig, ax = plt.subplots(figsize=(8,4))
agg_own["entity_accuracy"].plot(kind="bar", ax=ax, color="#2a9d8f")
ax.set_ylabel("Entity Accuracy"); ax.set_ylim(0,1); ax.set_title("Locality Entity Accuracy (your recordings)")
plt.xticks(rotation=20, ha="right"); plt.tight_layout(); plt.savefig("chart_entity.png", dpi=150); plt.show()

# WER on open data
if len(open_df):
    piv = open_df.pivot_table(index="model", columns="dataset", values="wer", aggfunc="mean")
    piv.plot(kind="bar", figsize=(8,4)); plt.ylabel("WER (lower=better)")
    plt.title("WER on open-source Hindi data"); plt.xticks(rotation=20, ha="right")
    plt.tight_layout(); plt.savefig("chart_wer_open.png", dpi=150); plt.show()

## 13 · Failure cases on your recordings (for the report)

In [ ]:
fails = own_df[~own_df["entity_matched"]][["model","item","condition","language","transcript","entity_score"]]
fails = fails.sort_values("entity_score")
fails.to_csv("failures.csv", index=False)
print(f"{len(fails)} failure rows. Worst 15:")
import pandas as pd
pd.set_option("display.max_colwidth", 60)
display(fails.head(15))

## 14 · Download everything

In [ ]:
from google.colab import files
import shutil, os
os.makedirs("bundle", exist_ok=True)
for f in ["results_own.csv","results_open.csv","agg_own.csv","agg_open.csv","failures.csv",
          "chart_entity.png","chart_wer_open.png"]:
    if os.path.exists(f): shutil.copy(f, "bundle/")
shutil.make_archive("asr_results", "zip", "bundle")
files.download("asr_results.zip")